# Módulo 1 · Fundamentos Matemáticos y Programación para IA

## Objetivos
Al terminar este notebook serás capaz de:

1. Manipular vectores, matrices y tensores con **NumPy** y entender qué operación de álgebra lineal hay detrás de cada línea de un modelo de ML.
2. Explicar y calcular **gradientes**, la **regla de la cadena** y el **descenso de gradiente**: el motor de todo el Deep Learning.
3. Descomponer una matriz con **eigen-descomposición** y **SVD**, y usar la SVD para implementar PCA desde cero.
4. Resolver numéricamente una **ecuación diferencial ordinaria (EDO)** y ver por qué las EDOs reaparecen en modelos modernos (Neural ODEs, *diffusion*, *flow matching*).
5. Aplicar **estadística computacional** (bootstrap, distribuciones muestrales, pruebas de permutación) en vez de fórmulas memorizadas.
6. Organizar datos tabulares con **pandas 2.x** (y saber cuándo usar **Polars**).

## Mapa de lectura (capítulos verificados)
| Tema | Libro | Capítulo verificado |
|---|---|---|
| Funciones de varias variables, derivadas parciales, gradiente, optimización | Stewart, *Cálculo de varias variables. Trascendentes tempranas* | **Cap. 14** *Derivadas parciales* (14.3 derivadas parciales, 14.6 gradiente, 14.7 máximos/mínimos, 14.8 multiplicadores de Lagrange) |
| Sistemas lineales, matrices, espacios vectoriales, producto interno, eigenvalores | Kolman & Hill, *Álgebra Lineal* (8.ª ed.) | **Cap. 1** Ecuaciones lineales y matrices · **Cap. 4** Espacios vectoriales reales · **Cap. 5** Espacios con producto interno · **Cap. 8** Valores y vectores propios |
| Álgebra lineal con Python orientada a datos | Cohen, *Practical Linear Algebra for Data Science* | **Cap. 2-3** Vectores · **Cap. 5-6** Matrices · **Cap. 11** Modelos lineales y mínimos cuadrados · **Cap. 13** Eigendecomposition · **Cap. 14** SVD |
| EDOs de primer orden y métodos numéricos | Zill, *Ecuaciones diferenciales con aplicaciones de modelado* | **Cap. 1** Introducción · **Cap. 2** EDO de primer orden · **Cap. 9** Soluciones numéricas |
| EDA, distribuciones muestrales, bootstrap, pruebas | Bruce, Bruce & Gedeck, *Practical Statistics for Data Scientists* (2.ª ed.) | **Cap. 1** Exploratory Data Analysis · **Cap. 2** Data and Sampling Distributions · **Cap. 3** Statistical Experiments and Significance Testing |
| Matemática para DL (repaso compacto) | Zhang et al., *Dive into Deep Learning* | **Cap. 2** Preliminaries · **Apéndice: Mathematics for Deep Learning** |

> **Nota de actualización (2026).** Las referencias de la versión anterior de la malla ("Stewart Cap. 4-5") apuntaban a los capítulos de cálculo de *una* variable; para IA lo relevante es el **Cap. 14** de la edición de varias variables. También se añadieron Cohen Cap. 11/13/14 (mínimos cuadrados, eigen, SVD), que son el corazón matemático de PCA, regresión y *embeddings*.

> **Ruta de aprendizaje Intensiva en ML Aplicado y DL Moderno** · Notebook del **Módulo 1 · Fundamentos Matemáticos y Programación para IA**
> Licencia CC BY-NC-SA 4.0 · © 2025-2026 Alexander Saravia
>
> Cómo usar este notebook: léelo de arriba hacia abajo. Cada sección tiene (1) la **idea** explicada con palabras y fórmulas,
> (2) **código ejecutable** que la materializa desde cero o con la librería estándar de 2026, (3) **lectura guiada** con
> el capítulo *verificado* de la bibliografía y (4) **ejercicios**. Las celdas marcadas `# OPCIONAL` requieren dependencias
> pesadas (GPU, Hugging Face, Spark) y están protegidas con `try/except` para que el resto del notebook siga corriendo.

## 0. Entorno de trabajo en 2026

El *stack* recomendado para todo el path:

* **Python 3.11–3.13** gestionado con [`uv`](https://docs.astral.sh/uv/) (sustituye a `pip`+`venv`+`pyenv` y es 10-100× más rápido).
* **NumPy 2.x** (nuevo `copy=` semántico, `np.strings`), **pandas 2.x** (backend Arrow opcional), **Polars** para datos grandes en una sola máquina.
* **Matplotlib 3.9+ / Seaborn 0.13** para gráficos; **Plotly** si necesitas interactividad.
* Cuadernos: **JupyterLab 4** o VS Code. Para GPU: CUDA 12.x (NVIDIA), Metal (Apple) o ROCm (AMD); PyTorch 2.x detecta el acelerador con `torch.accelerator`.

Instalación reproducible (ver `requirements.txt` del repositorio):

```bash
uv venv .venv && source .venv/bin/activate
uv pip install -r requirements.txt
```

In [ ]:
import sys, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("Python", sys.version.split()[0], "|", platform.machine())
print("NumPy", np.__version__, "| pandas", pd.__version__, "| Matplotlib", matplotlib.__version__)
rng = np.random.default_rng(42)   # generador moderno de NumPy (preferir a np.random.seed)

## 1. Vectores, matrices y tensores: el lenguaje de los datos

Un **vector** $\mathbf{x} \in \mathbb{R}^d$ es un punto (una observación con $d$ características). Una **matriz** $X \in \mathbb{R}^{n\times d}$ es un conjunto de $n$ observaciones apiladas. Un **tensor** generaliza a más ejes (imágenes: `alto × ancho × canales`; lotes de secuencias: `batch × tiempo × features`).

Las operaciones que aparecen en *todos* los modelos:

| Operación | Fórmula | NumPy | Dónde aparece |
|---|---|---|---|
| Producto punto | $\mathbf{a}\cdot\mathbf{b} = \sum_i a_i b_i$ | `a @ b` | similitud, neuronas, atención |
| Norma $\ell_2$ | $\|\mathbf{a}\|_2 = \sqrt{\mathbf{a}\cdot\mathbf{a}}$ | `np.linalg.norm(a)` | regularización, distancia |
| Similitud coseno | $\cos\theta = \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}$ | — | *embeddings*, RAG |
| Producto matriz-vector | $X\mathbf{w}$ | `X @ w` | regresión lineal, capa densa |
| Proyección de $\mathbf{b}$ sobre $\mathbf{a}$ | $\frac{\mathbf{a}\cdot\mathbf{b}}{\mathbf{a}\cdot\mathbf{a}}\mathbf{a}$ | — | mínimos cuadrados, PCA |

**Lectura:** Kolman & Hill Cap. 1 y 5; Cohen Cap. 2-3 y 5-6.

In [ ]:
a = np.array([3.0, 4.0])
b = np.array([1.0, 2.0])

punto = a @ b
norma_a = np.linalg.norm(a)
coseno = punto / (np.linalg.norm(a) * np.linalg.norm(b))
proy_b_sobre_a = (a @ b) / (a @ a) * a

print(f"a·b = {punto:.2f}   ||a|| = {norma_a:.2f}   cos(θ) = {coseno:.4f}")
print("proyección de b sobre a =", proy_b_sobre_a)

# Una 'capa densa' de una red neuronal es solo X @ W + b sobre un lote de datos
X = rng.normal(size=(5, 3))     # 5 observaciones, 3 features
W = rng.normal(size=(3, 2))     # 3 entradas -> 2 salidas
bias = np.zeros(2)
salida = X @ W + bias           # broadcasting del sesgo
print("forma de la salida:", salida.shape)

### Broadcasting y vectorización
La regla de oro en NumPy/PyTorch: **evita bucles en Python**. El *broadcasting* alinea las dimensiones desde la derecha y "estira" las de tamaño 1. Un ejemplo omnipresente: estandarizar columnas (restar la media y dividir por la desviación) de una matriz `n×d` con vectores de tamaño `d`.

In [ ]:
X = rng.normal(loc=[10, 200, -3], scale=[1, 50, 0.5], size=(1000, 3))
X_std = (X - X.mean(axis=0)) / X.std(axis=0)          # (1000,3) - (3,) -> broadcasting
print("medias ≈ 0:", np.round(X_std.mean(axis=0), 6))
print("desv   ≈ 1:", np.round(X_std.std(axis=0), 6))

# Distancias por pares sin bucles: ||x_i - x_j||^2 = ||x_i||^2 + ||x_j||^2 - 2 x_i·x_j
A = rng.normal(size=(4, 3))
sq = (A**2).sum(axis=1)
D2 = sq[:, None] + sq[None, :] - 2 * A @ A.T
print("matriz de distancias al cuadrado (4x4), diagonal ≈ 0:\n", np.round(D2, 3))

## 2. Cálculo: derivadas parciales, gradiente y regla de la cadena

Sea $f:\mathbb{R}^d\to\mathbb{R}$ (una **función de pérdida**). Su **gradiente** es el vector de derivadas parciales

$$\nabla f(\mathbf{w}) = \left(\frac{\partial f}{\partial w_1},\dots,\frac{\partial f}{\partial w_d}\right)^\top ,$$

y apunta en la dirección de **máximo crecimiento** (Stewart 14.6). Por eso el **descenso de gradiente** actualiza

$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta\,\nabla f(\mathbf{w}_t),$$

con $\eta$ la *tasa de aprendizaje*.

La **regla de la cadena** (Stewart 14.5) para composiciones $f(g(\mathbf{w}))$:
$$\frac{\partial f}{\partial w_i} = \sum_j \frac{\partial f}{\partial g_j}\frac{\partial g_j}{\partial w_i}.$$
Backpropagation (Módulo 3) **es** la regla de la cadena aplicada de forma ordenada sobre un grafo de cómputo.

Verificaremos gradientes analíticos con **diferencias finitas centradas**:
$$\frac{\partial f}{\partial w_i} \approx \frac{f(\mathbf{w}+h\mathbf{e}_i)-f(\mathbf{w}-h\mathbf{e}_i)}{2h}.$$
Este *gradient check* es la herramienta de depuración número uno cuando implementas algo desde cero.

In [ ]:
def f(w):
    # f(w) = (w0 - 3)^2 + 2*(w1 + 1)^2 + w0*w1   (cuenco elíptico con término cruzado)
    return (w[0] - 3) ** 2 + 2 * (w[1] + 1) ** 2 + w[0] * w[1]

def grad_f(w):
    return np.array([2 * (w[0] - 3) + w[1], 4 * (w[1] + 1) + w[0]])

def grad_numerico(f, w, h=1e-5):
    g = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w); e[i] = h
        g[i] = (f(w + e) - f(w - e)) / (2 * h)
    return g

w0 = np.array([0.5, -2.0])
print("analítico:", grad_f(w0))
print("numérico :", grad_numerico(f, w0))
print("error máx:", np.abs(grad_f(w0) - grad_numerico(f, w0)).max())

In [ ]:
def descenso_gradiente(grad, w, lr=0.1, pasos=60):
    trayectoria = [w.copy()]
    for _ in range(pasos):
        w = w - lr * grad(w)
        trayectoria.append(w.copy())
    return np.array(trayectoria)

# Óptimo analítico: resolver ∇f = 0  ->  sistema lineal 2x2
A = np.array([[2.0, 1.0], [1.0, 4.0]]); rhs = np.array([6.0, -4.0])
w_opt = np.linalg.solve(A, rhs)

fig, ax = plt.subplots(figsize=(6, 5))
g1, g2 = np.meshgrid(np.linspace(-2, 7, 200), np.linspace(-5, 3, 200))
Z = (g1 - 3) ** 2 + 2 * (g2 + 1) ** 2 + g1 * g2
ax.contour(g1, g2, Z, levels=30, cmap="Greys", alpha=0.7)
for lr, color in [(0.05, "tab:blue"), (0.2, "tab:orange"), (0.45, "tab:red")]:
    tr = descenso_gradiente(grad_f, np.array([-1.5, 2.5]), lr=lr)
    ax.plot(tr[:, 0], tr[:, 1], "o-", ms=3, color=color, label=f"lr={lr}")
ax.plot(*w_opt, "k*", ms=14, label="óptimo")
ax.set_title("Descenso de gradiente: la tasa de aprendizaje importa"); ax.legend(); plt.show()
print("óptimo:", w_opt)

**Observa:** con `lr=0.45` la trayectoria oscila (la matriz Hessiana tiene un autovalor grande en esa dirección). La condición de estabilidad para funciones cuadráticas es $\eta < 2/\lambda_{\max}(H)$: aquí conectan cálculo y álgebra lineal. En redes reales usamos tasas adaptativas (**Adam/AdamW**) y *schedulers* precisamente por este problema.

### Multiplicadores de Lagrange → regularización
Minimizar $f(\mathbf{w})$ sujeto a $\|\mathbf{w}\|_2^2 \le c$ (Stewart 14.8) equivale a minimizar $f(\mathbf{w}) + \lambda\|\mathbf{w}\|_2^2$: **Ridge** / *weight decay*. Con la norma $\ell_1$ obtenemos **Lasso**, cuya geometría en "diamante" produce soluciones dispersas (ISLP Cap. 6, Módulo 2).

## 3. Eigen-descomposición y SVD: la anatomía de una matriz

Para una matriz cuadrada $A$, un **vector propio** $\mathbf{v}$ satisface $A\mathbf{v} = \lambda\mathbf{v}$: la transformación solo lo *escala*. Si $A$ es simétrica (p. ej. una matriz de covarianza $\Sigma = \frac{1}{n}X^\top X$), entonces $A = Q\Lambda Q^\top$ con $Q$ ortogonal.

La **descomposición en valores singulares (SVD)** funciona para *cualquier* matriz:
$$X = U\,S\,V^\top,\qquad U^\top U = I,\ V^\top V = I,\ S=\mathrm{diag}(s_1\ge s_2\ge\dots).$$
* Las columnas de $V$ son las **direcciones principales** (PCA).
* $s_i^2/(n-1)$ es la varianza explicada por cada dirección.
* Truncar a $k$ valores singulares da la **mejor aproximación de rango $k$** (teorema de Eckart–Young): base de la compresión, los sistemas de recomendación y **LoRA** (Módulo 6).

**Lectura:** Kolman & Hill Cap. 8; Cohen Cap. 13-14; Bruce Cap. 7 (PCA).

In [ ]:
# Datos 2D correlacionados
n = 500
X = rng.multivariate_normal(mean=[0, 0], cov=[[3.0, 2.2], [2.2, 2.0]], size=n)
Xc = X - X.mean(axis=0)

# PCA vía eigen-descomposición de la covarianza
Sigma = Xc.T @ Xc / (n - 1)
lam, Q = np.linalg.eigh(Sigma)             # eigh: para matrices simétricas (ordena ascendente)
lam, Q = lam[::-1], Q[:, ::-1]

# PCA vía SVD de los datos centrados
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
var_svd = s**2 / (n - 1)

print("autovalores (eigh):", np.round(lam, 4))
print("varianzas (SVD)   :", np.round(var_svd, 4), "-> coinciden")
print("1ª dirección eigh :", np.round(Q[:, 0], 4))
print("1ª dirección SVD  :", np.round(Vt[0], 4), "(igual salvo signo)")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(Xc[:, 0], Xc[:, 1], s=8, alpha=0.4)
for i in range(2):
    v = Vt[i] * 2 * np.sqrt(var_svd[i])
    ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="->", lw=2.5, color=f"C{i+1}"))
ax.set_aspect("equal"); ax.set_title("Componentes principales = vectores singulares derechos"); plt.show()

In [ ]:
# Aproximación de bajo rango (Eckart-Young) sobre una 'imagen' sintética
img = np.zeros((64, 64))
yy, xx = np.mgrid[0:64, 0:64]
img += np.exp(-((xx - 20) ** 2 + (yy - 20) ** 2) / 80)
img += 0.8 * np.exp(-((xx - 45) ** 2 + (yy - 40) ** 2) / 200)
img[30:34, :] = 1.0
U, s, Vt = np.linalg.svd(img)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, k in zip(axes, [1, 3, 8, 64]):
    approx = (U[:, :k] * s[:k]) @ Vt[:k]
    err = np.linalg.norm(img - approx) / np.linalg.norm(img)
    ax.imshow(approx, cmap="gray"); ax.set_title(f"rango {k}, error {err:.1%}"); ax.axis("off")
plt.suptitle("SVD truncada: pocos valores singulares capturan casi toda la estructura"); plt.show()

## 4. Mínimos cuadrados: el modelo lineal desde el álgebra

Regresión lineal: encontrar $\mathbf{w}$ que minimiza $\|X\mathbf{w}-\mathbf{y}\|_2^2$. Igualando el gradiente a cero se obtienen las **ecuaciones normales**
$$X^\top X\,\mathbf{w} = X^\top\mathbf{y}\quad\Rightarrow\quad \hat{\mathbf{w}} = (X^\top X)^{-1}X^\top\mathbf{y}.$$
Geométricamente, $X\hat{\mathbf{w}}$ es la **proyección ortogonal** de $\mathbf{y}$ sobre el espacio columna de $X$ (Cohen Cap. 11). En la práctica *nunca* se invierte $X^\top X$: se usa la descomposición QR o la SVD (`np.linalg.lstsq`), numéricamente estables. Comparemos las tres rutas y el descenso de gradiente.

In [ ]:
n, d = 200, 3
X = np.c_[np.ones(n), rng.normal(size=(n, d))]     # columna de 1s = intercepto
w_true = np.array([2.0, -1.0, 0.5, 3.0])
y = X @ w_true + rng.normal(scale=0.5, size=n)

w_normal = np.linalg.solve(X.T @ X, X.T @ y)
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)

w_gd = np.zeros(d + 1)
for _ in range(500):
    w_gd -= 0.05 * (2 / n) * X.T @ (X @ w_gd - y)   # gradiente de la MSE

print("verdadero      :", w_true)
print("ec. normales   :", np.round(w_normal, 3))
print("lstsq (SVD/QR) :", np.round(w_lstsq, 3))
print("desc. gradiente:", np.round(w_gd, 3))

## 5. Ecuaciones diferenciales: de Zill a los modelos generativos

Una EDO de primer orden $\frac{dx}{dt} = v(x,t)$ describe cómo evoluciona un estado. El método de **Euler** (Zill Cap. 9.1) la resuelve avanzando en pasos pequeños:
$$x_{k+1} = x_k + h\,v(x_k, t_k).$$

¿Por qué importa esto en IA moderna?
* **Descenso de gradiente** es Euler aplicado al *flujo de gradiente* $\dot{\mathbf{w}} = -\nabla f(\mathbf{w})$.
* Una **red residual** $x_{l+1} = x_l + F(x_l)$ es un paso de Euler → **Neural ODEs**.
* Los modelos de **difusión** y **flow matching** (Módulo 4) generan datos integrando una EDO/SDE aprendida $\dot{x}=v_\theta(x,t)$ desde ruido hasta datos.

Resolvamos el crecimiento logístico $\dot{x} = r x(1-x/K)$ (Zill Cap. 3) con Euler y con `scipy.integrate.solve_ivp` (Runge–Kutta adaptativo).

In [ ]:
from scipy.integrate import solve_ivp

r, K = 1.2, 10.0
v = lambda t, x: r * x * (1 - x / K)
x_exacta = lambda t, x0: K / (1 + (K / x0 - 1) * np.exp(-r * t))

def euler(v, x0, t0, t1, h):
    ts = np.arange(t0, t1 + 1e-12, h); xs = [x0]
    for t in ts[:-1]:
        xs.append(xs[-1] + h * v(t, xs[-1]))
    return ts, np.array(xs)

t_fino = np.linspace(0, 8, 400)
sol = solve_ivp(v, (0, 8), [0.5], t_eval=t_fino, rtol=1e-8)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_fino, x_exacta(t_fino, 0.5), "k", lw=3, alpha=0.4, label="exacta")
for h in [1.0, 0.5, 0.1]:
    ts, xs = euler(v, 0.5, 0, 8, h)
    ax.plot(ts, xs, "o-", ms=3, label=f"Euler h={h}")
ax.plot(sol.t, sol.y[0], "--", label="solve_ivp (RK45)")
ax.set_xlabel("t"); ax.set_ylabel("x(t)"); ax.set_title("EDO logística: el paso h controla el error"); ax.legend(); plt.show()

## 6. Estadística computacional para científicos de datos

Bruce et al. (Cap. 2-3) proponen una idea liberadora: en vez de recordar fórmulas de errores estándar, **remuestrea**.

* **Distribución muestral**: repitiendo el muestreo, ¿cuánto varía un estadístico (media, mediana, coeficiente)? Con el **bootstrap** aproximamos esto remuestreando *con reemplazo* de la propia muestra.
* **Intervalo de confianza bootstrap**: percentiles 2.5 % y 97.5 % de los estadísticos remuestreados.
* **Prueba de permutación**: bajo $H_0$ las etiquetas son intercambiables; barajamos y vemos qué tan raro es el efecto observado. Es la forma más honesta de hacer un **test A/B**.

In [ ]:
# Ingresos sesgados (lognormal): la media es sensible, la mediana robusta
ingresos = rng.lognormal(mean=10, sigma=0.8, size=300)

def bootstrap(stat, x, B=4000):
    idx = rng.integers(0, len(x), size=(B, len(x)))
    return stat(x[idx], axis=1)

boot_media = bootstrap(np.mean, ingresos)
boot_mediana = bootstrap(np.median, ingresos)
ic = lambda b: np.percentile(b, [2.5, 97.5])
print(f"media   = {ingresos.mean():,.0f}  IC95% bootstrap = {ic(boot_media).round(0)}")
print(f"mediana = {np.median(ingresos):,.0f}  IC95% bootstrap = {ic(boot_mediana).round(0)}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(ingresos, bins=40, color="gray"); axes[0].set_title("muestra (asimétrica)")
axes[1].hist(boot_media, bins=40, alpha=0.7, label="media"); axes[1].hist(boot_mediana, bins=40, alpha=0.7, label="mediana")
axes[1].set_title("distribuciones muestrales bootstrap"); axes[1].legend(); plt.show()

In [ ]:
# Test A/B con prueba de permutación: tasa de conversión de dos variantes de una página
conv_A = rng.binomial(1, 0.110, size=2000)
conv_B = rng.binomial(1, 0.125, size=2000)
efecto_obs = conv_B.mean() - conv_A.mean()

todos = np.concatenate([conv_A, conv_B])
def perm_stat():
    p = rng.permutation(todos)
    return p[2000:].mean() - p[:2000].mean()
perm = np.array([perm_stat() for _ in range(5000)])
p_valor = (np.abs(perm) >= abs(efecto_obs)).mean()

print(f"efecto observado B-A = {efecto_obs:.4f}   p-valor permutación (bilateral) = {p_valor:.4f}")
plt.hist(perm, bins=50, color="lightgray"); plt.axvline(efecto_obs, color="red", lw=2, label="observado")
plt.title("Distribución nula por permutación"); plt.legend(); plt.show()

> **Trampa clásica (Bruce Cap. 3):** *p-hacking* y pruebas múltiples. Si pruebas 20 métricas al 5 %, esperas 1 falso positivo. Ajusta (Bonferroni, Benjamini–Hochberg; ISLP Cap. 13) o **pre-registra** la métrica primaria.

## 7. Datos tabulares: pandas 2.x y Polars

pandas sigue siendo el estándar para exploración; en 2026 la práctica recomendada es:
* usar `pd.read_csv(..., dtype_backend="pyarrow")` para tipos Arrow (menos memoria, *strings* rápidos),
* encadenar operaciones con `.pipe()` y `.assign()` en vez de mutaciones,
* migrar a **Polars** (API perezosa, multihilo) o **DuckDB** cuando los datos superan la RAM cómoda (Módulo 9).

Practicaremos el flujo de EDA de Bruce Cap. 1: tipos de variables, estimadores de localización y dispersión, percentiles, correlación y visualización.

In [ ]:
n = 1000
df = pd.DataFrame({
    "region": rng.choice(["norte", "sur", "este", "oeste"], size=n, p=[0.4, 0.3, 0.2, 0.1]),
    "edad": rng.integers(18, 80, size=n),
    "ingreso": rng.lognormal(10, 0.6, size=n).round(0),
})
df["gasto"] = (0.35 * df["ingreso"] * rng.uniform(0.7, 1.3, size=n) + 50 * (df["edad"] > 50)).round(0)

resumen = (df.groupby("region", observed=True)
             .agg(n=("edad", "size"),
                  ingreso_mediano=("ingreso", "median"),
                  gasto_medio=("gasto", "mean"),
                  gasto_iqr=("gasto", lambda s: s.quantile(0.75) - s.quantile(0.25)))
             .sort_values("n", ascending=False))
display(resumen)

print("correlación Pearson ingreso~gasto :", round(df["ingreso"].corr(df["gasto"]), 3))
print("correlación Spearman ingreso~gasto:", round(df["ingreso"].corr(df["gasto"], method="spearman"), 3))
df.plot.scatter(x="ingreso", y="gasto", alpha=0.3, s=8, logx=True, figsize=(6, 4), title="ingreso vs gasto"); plt.show()

In [ ]:
# OPCIONAL: la misma agregación en Polars (sintaxis de expresiones, ejecución perezosa multihilo)
try:
    import polars as pl
    lf = pl.from_pandas(df).lazy()
    out = (lf.group_by("region")
             .agg(pl.len().alias("n"), pl.col("ingreso").median().alias("ingreso_mediano"),
                  pl.col("gasto").mean().alias("gasto_medio"))
             .sort("n", descending=True))
    print(out.collect())
except ImportError:
    print("Polars no instalado: `uv pip install polars`. El resto del notebook no lo necesita.")

## 8. Ejercicios

1. **Gradient check.** Implementa $f(\mathbf{w}) = \log\left(1+e^{\mathbf{x}\cdot\mathbf{w}}\right)$ (la pérdida logística para una observación) y su gradiente analítico $\sigma(\mathbf{x}\cdot\mathbf{w})\,\mathbf{x}$. Verifícalo con `grad_numerico`.
2. **Condicionamiento.** Genera $X$ con dos columnas casi colineales y compara `np.linalg.solve(X.T@X, X.T@y)` con `np.linalg.lstsq`. Calcula `np.linalg.cond(X)`. ¿Qué pasa al añadir Ridge ($X^\top X + \lambda I$)?
3. **PCA real.** Carga `sklearn.datasets.load_digits()`, centra, aplica SVD y grafica la varianza explicada acumulada. ¿Cuántas componentes explican el 90 %? Reconstruye un dígito con 10, 20 y 40 componentes.
4. **EDO → red residual.** Implementa una "red" que aplique 50 pasos de Euler a $\dot{x} = \tanh(Wx)$ con $W$ fija. Observa cómo transforma una nube de puntos 2D. Esto es una Neural ODE sin entrenar.
5. **Bootstrap de un coeficiente.** Reutiliza el ejemplo de mínimos cuadrados: remuestrea filas 2000 veces y construye el IC 95 % de $w_3$. Compáralo con el error estándar clásico $\sigma\sqrt{[(X^\top X)^{-1}]_{33}}$.
6. **Múltiples pruebas.** Simula 20 tests A/B donde $H_0$ es cierta. ¿Cuántos p-valores < 0.05 obtienes? Aplica la corrección de Benjamini–Hochberg (`scipy.stats.false_discovery_control`).

## Referencias
* Stewart, J. *Cálculo de varias variables. Trascendentes tempranas* (7.ª/8.ª ed.). Cengage. Cap. 14.
* Kolman, B. & Hill, D. R. *Álgebra Lineal* (8.ª ed.). Pearson. Cap. 1, 4, 5, 8.
* Cohen, M. X. (2022). *Practical Linear Algebra for Data Science*. O'Reilly. Cap. 2-3, 5-6, 11, 13-14. Código: github.com/mikexcohen/LinAlg4DataScience
* Zill, D. G. *Ecuaciones diferenciales con aplicaciones de modelado* (9.ª/10.ª ed.). Cengage. Cap. 1-3, 9.
* Bruce, P., Bruce, A. & Gedeck, P. (2020). *Practical Statistics for Data Scientists* (2.ª ed.). O'Reilly. Cap. 1-3. Código: github.com/gedeck/practical-statistics-for-data-scientists
* Zhang, A., Lipton, Z., Li, M. & Smola, A. *Dive into Deep Learning*. d2l.ai. Cap. 2 y Apéndice matemático.
* Chen, R. T. Q. et al. (2018). *Neural Ordinary Differential Equations*. NeurIPS.